# Exercise 6: evaluate

First read the errors and name them, then measure.

1. Read the wrong rows, name each failure
2. Count the failure types, then the accuracy per field
3. A second model as a judge, checked against your own labels

Run the cells one by one, from the top: **Shift+Enter**. After you change a cell, run it again, then the cells below it.

In [ ]:
from collections import Counter

import pandas as pd
from IPython.display import Markdown

import checks
from course import DATA, OUTPUT, ask, read_csv

pd.set_option("display.max_colwidth", 200)

**Choose the extraction to evaluate**: `"reference"`, a small model (llama3.2:3b), or `"mine"`, your extraction of exercise 5 (part 3 of exercise 5 saves it).

In [ ]:
source = "reference"

In [ ]:
files = {"reference": DATA / "extraction_reference.csv", "mine": OUTPUT / "extraction.csv"}
extracted = {row["id"]: row for row in read_csv(files[source])}
truth = {row["id"]: row for row in read_csv(DATA / "ground_truth.csv")}
Markdown(f"Ground truth: {len(truth)} abstracts, written by a person. A large model made no error on these in our test, so start with the small model")

## 1 · Read and name

> **The idea:** before you measure, read the errors and give them a name (slide "Lire les erreurs, comme pour MNIST")

The rows where an exact field is wrong. The `note` of the ground truth gives the reason.

In [ ]:
EXACT = ["sample_size", "study_design", "effect_found"]
wrong = [
    {"id": item_id, "field": field, "extracted": extracted[item_id].get(field, ""), "right": truth[item_id][field], "note": truth[item_id]["note"]}
    for item_id in truth if item_id in extracted
    for field in EXACT if str(extracted[item_id].get(field, "")).strip().lower() != truth[item_id][field].strip().lower()
]
pd.DataFrame(wrong[:5])

**Give a short name to each wrong row**: what type of failure is it? For example "clusters counted instead of people". One name per row of the table, in the same order.

In [ ]:
names = ["", "", "", "", ""]

In [ ]:
failure = {f"{row['id']} {row['field']}": name for row, name in zip(wrong[:5], names)}
checks.ex6_names(failure)

<details><summary>Question</summary>

Which rows have the same problem?

</details>

## 2 · Count, then measure

> **The idea:** accuracy says how many errors, the failure types say what to fix (slide "Que mesurer ?")

In [ ]:
counts = Counter(failure.values())
pd.DataFrame([{"failure type": name, "rows": count} for name, count in counts.most_common() if name])

**Choose a field**: `"sample_size"`, `"study_design"` or `"effect_found"`.

In [ ]:
field = "sample_size"

In [ ]:
right = sum(str(extracted[item_id].get(field, "")).strip().lower() == truth[item_id][field].strip().lower() for item_id in truth if item_id in extracted)
Markdown(f"**{field}: {right} / {len(truth)} correct**")

- Accuracy says *how often*. The list of failures says *what to fix*
- These 10 rows are a dev set: you look at them and change the prompt. A real test set stays closed until the end, as in the ML module

<details><summary>Question</summary>

Which one tells you how to change the prompt?

</details>

## 3 · A judge for one failure type

> **The idea:** an LLM judge is a classifier. Check that it agrees with you before you trust it (slide "Le juge est un classifieur : testez-le")

The field `main_result` is free text: a program cannot compare it word by word. A second model (Gemini) judges it.

In [ ]:
sample = [item_id for item_id in truth if item_id in extracted][:5]
pd.DataFrame([{"id": item_id, "extracted": extracted[item_id].get("main_result", ""), "ground truth": truth[item_id]["main_result"]} for item_id in sample])

**First, label the 5 rows yourself**: `True` when the extracted result is good, `False` when it is not. One per row, in the same order.

In [ ]:
my_labels = [True, True, True, True, True]

**Change one line of the rubric**, then run the two cells. Target: a judge that agrees with you.

In [ ]:
rubric = """Does the extracted result state the primary outcome and its direction, as the ground truth does?
Give your reason in one sentence first. Then, on the last line, write only PASS or FAIL."""

In [ ]:
my_label = dict(zip(sample, my_labels))
verdicts = {}
for item_id in sample:
    reply, _ = ask(f"{rubric}\n\nGround truth: {truth[item_id]['main_result']}\nExtracted: {extracted[item_id].get('main_result', '')}", "judge", temperature=0)
    verdicts[item_id] = reply.strip().splitlines()[-1].strip().upper().startswith("PASS")
failures = [item_id for item_id in sample if not my_label[item_id]]
good = [item_id for item_id in sample if my_label[item_id]]
display(Markdown(f"**The judge found {sum(not verdicts[i] for i in failures)} of {len(failures)} failure(s), and passed {sum(verdicts[i] for i in good)} of {len(good)} good answer(s)**"))
pd.DataFrame([{"id": i, "you": "good" if my_label[i] else "failure", "judge": "PASS" if verdicts[i] else "FAIL"} for i in sample])

**Key points**

- Read, name, count: then measure
- A judge is a test that you must test
- 10 rows is few: a real error analysis reads about 100

<details><summary>Question</summary>

Find a row where you and the judge disagree. Who is right?

</details>

---
## Option

The search of exercise 4: open its option and compare hit@3 for k = 1, 3, 5 passages.